# Plan 004.4.1 — Duplicate group sorting and streaming

**Design preview — API not implemented yet.** These examples specify the active [plan](../../docs/plans/004.4.1-duplicate-group-sorting-and-streaming.md). Code is displayed as Markdown deliberately: running this notebook does not execute proposed methods. Implementation must convert the examples to executable cells and validate them from a clean kernel.

Audience: users familiar with catalog scans and the Plan 004.4 duplicate tutorial. Prerequisite for execution: implementation of 004.4.1.

Learn to sort groups, distinguish group and member bounds, process all files incrementally, and close partially consumed iterators. This read interface will support later duplicate-handling workflows.

## 1. Disposable setup

Run these snippets in order once the API is implemented. The fixture has three groups: four files of size 4, three files of size 2, and two empty files. Expected totals: 3 groups, 9 files, 16 redundant bytes.

```python
from contextlib import closing
from pathlib import Path
from tempfile import TemporaryDirectory

from archiver import Catalog
from archiver.query import duplicate, size

workspace = TemporaryDirectory(prefix="archiver-004-4-1-")
base = Path(workspace.name)
root = base / "source"
root.mkdir()
for folder, payload, count in (
    ("large", b"AAAA", 4),
    ("small", b"BB", 3),
    ("empty", b"", 2),
):
    directory = root / folder
    directory.mkdir()
    for number in range(count):
        (directory / f"{number}.bin").write_bytes(payload)

catalog = Catalog.create(base / "catalog.sqlite")
catalog.scan_directory(root)
query = catalog.duplicates(root).sort(size.desc()).limit(2).member_limit(2)
```

## 2. Bounded fetch remains available

Fetch produces existing value objects with bounded member tuples. Streaming returns handles that load members only when asked.

```python
views = query.fetch()
assert len(views) == 2
assert [view.size_bytes for view in views] == [4, 2]
assert all(len(view.members) == 2 for view in views)
assert [view.file_instance_count for view in views] == [4, 3]
```

Group metrics are complete even when only two member previews are present.

## 3. Bounded streaming at both levels

The outer stream respects limit(2); each member stream respects member_limit(2).
closing makes resource cleanup explicit, including if processing raises an error.

```python
group_count = member_count = 0
with closing(query.stream()) as groups:
    for group in groups:
        group_count += 1
        with closing(group.stream_members()) as members:
            for member in members:
                member_count += 1
                print(group.size_bytes, member.relative_path)
assert (group_count, member_count) == (2, 4)
```

No full group collection is constructed. Each yielded handle has content_id,
size_bytes, file_instance_count, and potential_redundant_bytes.

## 4. Stream all groups and all files

Each explicit all-terminal removes only its corresponding limit. The same
query still has limits of two; neither applies in this example.

```python
group_count = member_count = redundant_bytes = 0
with closing(query.stream_all()) as groups:
    for group in groups:
        group_count += 1
        redundant_bytes += group.potential_redundant_bytes
        with closing(group.stream_all_members()) as members:
            for member in members:
                member_count += 1
                print(group.content_id, member.relative_path)
assert (group_count, member_count, redundant_bytes) == (3, 9, 16)
```

Omitting member_limit(2) would restore the default of 20; it would not remove
the bound. stream_all_members() explicitly streams every member regardless of
that setting. SQLite may need to aggregate and sort before the first yield;
the promise is incremental Python materialization, not constant SQLite memory.

## 5. Choose group and member bounds independently

All members of only the two selected groups:

```python
member_count = 0
with closing(query.stream()) as groups:
    for group in groups:
        with closing(group.stream_all_members()) as members:
            member_count += sum(1 for _ in members)
assert member_count == 7
```

Bounded members of every group:

```python
member_count = 0
with closing(query.stream_all()) as groups:
    for group in groups:
        with closing(group.stream_members()) as members:
            member_count += sum(1 for _ in members)
assert member_count == 6
```

## 6. Typed group ordering and early closure

```python
by_count = query.sort(duplicate.file_count.desc())
by_redundancy = query.sort(duplicate.redundant_bytes.desc())
smallest_first = query.sort(size.asc())

with closing(smallest_first.stream_all()) as groups:
    for group in groups:
        assert group.size_bytes == 0
        with closing(group.stream_all_members()) as members:
            first_member = next(members)
            print(first_member.relative_path)
        break
```

The most recent sort replaces the previous one. Ties always use full content
algorithm and digest ascending. Member paths are always ascending. A plain break
does not guarantee iterator closure, hence closing at both levels.

## 7. Exercise and answer

Predict the number of files visited when streaming every group but at most one
member per group. Does the complete file count on each handle change?

```python
visited = complete_file_count = 0
with closing(query.member_limit(1).stream_all()) as groups:
    for group in groups:
        complete_file_count += group.file_instance_count
        with closing(group.stream_members()) as members:
            visited += sum(1 for _ in members)
assert visited == 3
assert complete_file_count == 9
```

Optional extension after implementation: create more than 20 groups and a group
with more than 20 members to demonstrate the default bounds.

## 8. Scan lifetime, future workflows, and cleanup

A group stream pins one successful scan on first advancement. Handles retain
that scan for member queries even after refresh or outer-stream closure.
The catalog must remain open until all member work is finished.

These examples inspect observations. Later duplicate-handling plans must define
how to check that live files still match the observations, authorize actions,
and recover from partial failures before introducing file mutation.

```python
catalog.close()
workspace.cleanup()
```

During implementation, convert the fenced Python snippets to code cells, run
top-to-bottom in a clean kernel, validate all assertions, and strip outputs and
execution counts. This preview has been checked structurally; the proposed
streaming examples cannot be executed against the current API.